### Function to find kmers in sequence

In [2]:
import math

def find_kmers(sequence):
    sequence = sequence.upper()
    kmers_dict = {}
    l = len(sequence)
    for k in range(1, 8):
        max_kmers = l - k + 1
        kmers = set()
        for i in range(max_kmers):
            kmer = sequence[i:i + k]
            kmers.add(kmer)
        if 4**k <= max_kmers:
            kmers_dict[k] = len(kmers) / (4**k)
        else:
            kmers_dict[k] = len(kmers) / max_kmers
    return kmers_dict

#### Retrieve the whole Chr1 from hg38 genome

In [3]:
# Creating "data_assignment/" directory
import os

path = "./data_assignment"
# Check if path exists
if not os.path.exists(path):
    # Create the directory
    os.makedirs(path)
    print(path, "directory created")
else:
    print(path, "directory exists")

./data_assignment directory exists


In [4]:
# Check if human Chr1 hg38 file exists
fasta_path = './data_assignment/chr1.fa.gz'
if not os.path.exists(fasta_path):
    print("File does not exist. Downloading fasta file.")
    os.system('wget -q -O ./data_assignment/chr1.fa.gz "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr1.fa.gz"')
else:
    print("File already exists. Skipping download.")

File does not exist. Downloading fasta file.


In [5]:
!gunzip -k ./data_assignment/chr1.fa.gz

In [6]:
# Read FASTA and clean sequence
with open("./data_assignment/chr1.fa", 'r') as file:
    raw = file.read().upper()
    dna_seq = ''.join(base for base in raw if base in {'A', 'T', 'C', 'G'})

### Calculate complexity of whole Chr1

In [ ]:
# Sliding window parameters
window_size = 1000
step = 100

# Calculate complexity
complexity_scores_Chr1 = []
positions =[]
for j in range(0, len(dna_seq) - window_size + 1, step):
    window= dna_seq[j:j+window_size]
    kmers = find_kmers(window)
    complexity_values = list(kmers.values())
    complexity_score = math.prod(complexity_values)
    
    positions.append(j+window_size) # check ANTONIAS
    complexity_scores_Chr1.append(complexity_score)

median_score_Chr1 = statistics.median(complexity_scores_Chr1)
print(f"Median complexity score: {median_score_Chr1:.6f}") #print median complexity rounded in 6 decimals

### Plot complexity of the whole chromosome

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(positions, complexity_scores_Chr1, color='blue', linewidth=1)
plt.title("K-mer Complexity Across chr1")
plt.xlabel("Genome Position (start of 1000bp window)")
plt.ylabel("Complexity Score")
plt.grid(True)
plt.tight_layout()
plt.show()


### Obtain genome annotations

From the plot using all the sequence of chr1 we observed that around the positions ~121,000,000 to ~125,000,000 bp (GRCh38), there is a sharp reduce in complexity and we confirm from the genome browser of UCSC that this region in chromosome 1 in hg38 is correspond to centromere, which is known for a lot of repeats elements and low complexity

#### - Obtain FASTA file with CDS annotation (ENSEMBL)

In [9]:
# Check if human Chr1 with CDS hg38 anottation file exists
cds_fasta_path = './data_assignment/Homo_sapiens.GRCh38.cds.all.fa.gz'
if not os.path.exists(cds_fasta_path):
    print("File does not exist. Downloading fasta file.")
    os.system('wget -q -O ./data_assignment/Homo_sapiens.GRCh38.cds.all.fa.gz "http://ftp.ensembl.org/pub/release-114/fasta/homo_sapiens/cds/Homo_sapiens.GRCh38.cds.all.fa.gz"')
else:
    print("File already exists. Skipping download.")

File does not exist. Downloading fasta file.


In [10]:
#parse the file and filter for chromosome 1

from Bio import SeqIO
import gzip

chr1_cds_path = './data_assignment/chr1_cds.fa'

with gzip.open(cds_fasta_path, 'rt') as handle, open(chr1_cds_path, 'w') as out_handle:
    for record in SeqIO.parse(handle, 'fasta'):
        if 'chromosome:GRCh38:1:' in record.description:
            SeqIO.write(record, out_handle, 'fasta')

print(f"Saved chr1 CDS sequences to {chr1_cds_path}")

Saved chr1 CDS sequences to ./data_assignment/chr1_cds.fa


In [6]:
# Read FASTA and clean sequence
with open("./data_assignment/chr1_cds.fa", 'r') as file:
    raw = file.read().upper()
    dna_seq = ''.join(base for base in raw if base in {'A', 'T', 'C', 'G'})

### Calculate complexity of Chr1 CDS regions

In [7]:
# Sliding window parameters
window_size = 1000
step = 100

# Calculate complexity
complexity_scores_cds = []
positions =[]
for j in range(0, len(dna_seq) - window_size + 1, step):
    window= dna_seq[j:j+window_size]
    kmers = find_kmers(window)
    complexity_values = list(kmers.values())
    complexity_score = math.prod(complexity_values)
    
    positions.append(j+window_size) # check ANTONIAS
    complexity_scores_cds.append(complexity_score)

median_score_Chr1 = statistics.median(complexity_scores_cds)
print(f"Median complexity score: {median_score_cds:.6f}") #print median complexity rounded in 6 decimals

#### - Obtain FASTA file with Repeat Sequence annotation (RepeatMasker from UCSC)

In [4]:
# Check if FASTA file with repeats of human Chr1 hg38 exists
repeats_fasta_path = './data_assignment/chr1_repeats.fa.gz'
if not os.path.exists(repeats_fasta_path):
    print("File does not exist. Downloading fasta file.")
    os.system('wget -q -O ./data_assignment/chr1_repeats.fa.gz "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/chromFaMasked/chr1.fa.gz"')
else:
    print("File already exists. Skipping download.")

File does not exist. Downloading fasta file.


The aforementioned file contains **lowercase = repeat-masked** sequence for chr1. The uppercase bases represent non-repetitive regions.

For this reason, we need the following code to extract **only the repetitive sequences**.


In [5]:
#Extract repetitive sequences
from Bio import SeqIO
import gzip

repeat_only_path = './data_assignment/chr1_repeats_only.fa'

with gzip.open(repeats_fasta_path, 'rt') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        repeat_seq = ''.join([b for b in str(record.seq) if b.islower()])
        with open(repeat_only_path, 'w') as out:
            out.write(f">{record.id}_repeats\n{repeat_seq}\n")

print(f"Saved repeat-only sequence to {repeat_only_path}")

In [6]:
# Read FASTA and clean sequence
with open("./data_assignment/chr1_repeats_only.fa", 'r') as file:
    raw = file.read().upper()
    dna_seq = ''.join(base for base in raw if base in {'A', 'T', 'C', 'G'})

### Calculate complexity of Chr1 repetitive regions

In [7]:
# Sliding window parameters
window_size = 1000
step = 100

# Calculate complexity
complexity_scores_repeat = []
positions =[]
for j in range(0, len(dna_seq) - window_size + 1, step):
    window= dna_seq[j:j+window_size]
    kmers = find_kmers(window)
    complexity_values = list(kmers.values())
    complexity_score = math.prod(complexity_values)
    
    positions.append(j+window_size) # check ANTONIAS
    complexity_scores_repeat.append(complexity_score)

median_score_Chr1 = statistics.median(complexity_scores_cds)
print(f"Median complexity score: {median_score_cds:.6f}") #print median complexity rounded in 6 decimals

### Statistic test to find difference between complexity of CDS regions and repetitive regions

In [ ]:
from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(complexity_scores_repeats_regions, complexity_scores_coding_regions, alternative='two-sided')
print("Mann–Whitney U test results:")
print(f"U statistic = {stat:.4f}")
print(f"p-value     = {p:.4e}")

### Plot complexity of Chr1 CDS -VS- Repeats regions

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(data=[complexity_scores_repeat, complexity_scores_cds])
plt.xticks([0, 1], ['Repeats', 'CDs'])
plt.ylabel('Complexity Score')
plt.title('Comparison of Complexity: Repeats Vs CDs')
plt.show()
